#  A8: Ensemble Learning for Complex Regression Modeling on Bike Share Data 

## Part A: Data Preprocessing and Baseline 

### Data Loading and Feature Engineering

We will first load and view the data

In [1]:
import numpy as np
import pandas as pd
df=pd.read_csv('hour.csv')
df.head(5)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


We will now drop irrelevant columns like instant, dteday, casual and registered.

We then identify the categorical columns (season, yr, mnth, hr , holiday, weekday, workingday, weathersit)

In [2]:
df = df.drop(columns=['instant', 'dteday', 'casual', 'registered'])

# One-Hot Encoding for categorical columns
categorical_columns = ['season','yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit']
df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

# Display the first few rows
df.head()

,temp,atemp,hum,windspeed,cnt,season_2,season_3,season_4,yr_1,mnth_2,...,weekday_1,weekday_2,weekday_3,weekday_4,weekday_5,weekday_6,workingday_1,weathersit_2,weathersit_3,weathersit_4
0,0.24,0.2879,0.81,0.0,16,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
1,0.22,0.2727,0.80,0.0,40,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,0.22,0.2727,0.80,0.0,32,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
3,0.24,0.2879,0.75,0.0,13,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
4,0.24,0.2879,0.75,0.0,1,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False


We split the feature variable and the target variable

In [3]:
X = df.drop('cnt', axis=1)
y = df['cnt']



We now split the entire dataset into train and test dataset (20% of the entire dataset is used for test dataset).

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


We now train a single Decision Tree Regressor with max depth of 6 and a single Linear Regression model on the training data. 

In [5]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Decision Tree Regressor
dt_model = DecisionTreeRegressor(max_depth=6, random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))

# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print(f'Decision Tree RMSE: {rmse_dt:.2f}')
print(f'Linear Regression RMSE: {rmse_lr:.2f}')


Decision Tree RMSE: 118.46
Linear Regression RMSE: 100.45


The Linear Regression model has the smaller RMSE so we use that as the baseline model.

## Part B: Ensemble Techniques for Bias and Variance Reduction

### Bagging

We will now implement a Bagging Regressor using the Decision Tree Regressor as the base estimator with hyperparameter tuning.

In [19]:
import numpy as np
import pandas as pd
import warnings
from scipy.stats import randint, uniform
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import FitFailedWarning


# Base Bagging estimator 
base_tree = DecisionTreeRegressor(random_state=42)
bag = BaggingRegressor(estimator=base_tree, random_state=42, n_jobs=-1)

# Parameter distributions
param_distributions = {
    "n_estimators": randint(50, 301),       
    "max_samples": uniform(0.5, 0.5),         
    "max_features": uniform(0.5, 0.5),       
    "bootstrap": [True, False],
    "bootstrap_features": [True, False],
    "estimator__max_depth": randint(2, 30),   
    "estimator__min_samples_split": randint(2, 41),
    "estimator__min_samples_leaf": randint(1, 41),
    "estimator__max_features": [None, "sqrt", "log2"] 
}

# Randomized search
rnd_search = RandomizedSearchCV(
    estimator=bag,
    param_distributions=param_distributions,
    n_iter=5,                               
    scoring="neg_mean_squared_error",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=False,
    error_score=np.nan                        
)

# Run search on training data
rnd_search.fit(X_train, y_train)

# Best params & CV RMSE
best_params = rnd_search.best_params_
best_cv_rmse = np.sqrt(-rnd_search.best_score_)


# Evaluate best estimator on the test set
best_bag = rnd_search.best_estimator_
y_pred_test = best_bag.predict(X_test)
rmse_bagging = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f"Bagging RMSE: {rmse_bagging:.4f}")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Bagging RMSE: 101.4634


We can see that RMSE after bagging is less than of the desicion tree model.

Bagging  is used to reduce model variance by averaging the predictions of multiple models trained on different random subsets of the data. Each Decision Tree in the ensemble sees a slightly different sample of the training set (drawn with replacement), so their individual errors tend to cancel out when averaged leading to a model with less variance.

### Boosting

We will now implement a Gradient Boosting Regressor with hyperparameter tuning.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from scipy.stats import randint, uniform

# Parameters
param_distributions = {
    "n_estimators": randint(100, 1001),        
    "learning_rate": uniform(0.01, 0.49),      
    "max_depth": randint(2, 8),                
    "min_samples_split": randint(2, 21),
    "min_samples_leaf": randint(1, 21),
    "subsample": uniform(0.5, 0.5),            
    "max_features": [None, "sqrt", "log2"]
}

# Base estimator
gbr = GradientBoostingRegressor(random_state=42)

# Randomized search
rnd_search = RandomizedSearchCV(
    estimator=gbr,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="neg_mean_squared_error", 
    cv=5,
    verbose=0,
    random_state=42,
    n_jobs=-1,
    error_score=np.nan,
    return_train_score=False
)

# Run search on training data
rnd_search.fit(X_train, y_train)


best_params = rnd_search.best_params_
best_neg_mse = rnd_search.best_score_
best_cv_rmse = np.sqrt(-best_neg_mse)


# Evaluate best estimator on the test set
best_gbr = rnd_search.best_estimator_
y_pred_test = best_gbr.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

# fit on full training data with best params
final_model_gboost = GradientBoostingRegressor(**best_params, random_state=42)
final_model_gboost.fit(X_train, y_train)
y_pred_test_final = final_model_gboost.predict(X_test)
rmse_gboost = np.sqrt(mean_squared_error(y_test, y_pred_test))
print("Gradient Boosting RMSE:",rmse_gboost)


Gradient Boosting RMSE: 44.43127070136962


We can see that RMSE after Boosting is much better than the previous models.



The Gradient Boosting Regressor is designed to reduce bias by sequentially building a strong model from multiple weak learners. Each new tree in the sequence focuses on correcting the error made by the previous ensemble hence it will help in reducing bias.

In [8]:
print(f'Baseline Linear Regression RMSE: {rmse_lr:.2f}')
print(f'Bagging Regressor RMSE: {rmse_bagging:.2f}')
print(f'Gradient Boosting Regressor RMSE: {rmse_gboost:.2f}')


Baseline Linear Regression RMSE: 100.45
Bagging Regressor RMSE: 101.46
Gradient Boosting Regressor RMSE: 44.43


## Part C: Stacking for Optimal Performance 

### Stacking Implementation

#### Stacking

Stacking is an ensemble learning technique that combines predictions of different base models (level-0 models) by another meta model (level-1 model) such that we only capture the strength of each model.

Each base model is trained independently on the same training data. But since the different models use different learning algorithms, they tend to capture different patterns in the data and make different types of errors. Stacking helps in capturing the strength of each model.

#### Base Learners

- **K-Nearest Neighbors (KNN) regressor**:- It is a model that predicts the target value for a data point by averaging the target values of its k closest samples in the feature space.
- **Bagging Regressor (Bootstrap Aggregating)**:- It is an ensemble method that trains multiple instances of a base model on different random subsets of the training data (data is sampled with replacement).
- **Gradient Boosting Regressor**:-It is a boosting based ensemble that builds trees sequentially, where each new tree learns to correct the residual errors of the previous ensemble.

#### Meta Learner

The Meta Learner in a stacking ensemble is the model that learns how to combine the predictions of multiple base learners to produce a final improved output. In ridge regression we introduce L2 norm term for the loss function thus preventing any single coefficient from dominating the final model thus reducing overfitting and improving stability.

#### Stack Regressor Implementation

Training KNN with hyper parameter tuning.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
import numpy as np

knn_pipe = Pipeline([
    ("scaler", StandardScaler()),         
    ("knn", KNeighborsRegressor(n_neighbors=5))
])
#Hyperparameter tuning 
knn_param_grid = {
    "knn__n_neighbors": [3, 5, 7, 10, 15],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]   
}

knn_search = GridSearchCV(
    estimator=knn_pipe,              
    param_grid=knn_param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

knn_search.fit(X_train, y_train)

best_knn = knn_search.best_estimator_



Fitting 5 folds for each of 20 candidates, totalling 100 fits


Tuning the stacking regressor.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import BaggingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
import numpy as np

base_stack = StackingRegressor(
    estimators=[
        ("knn", best_knn),
        ("bagging", best_bag),
        ("gboost", final_model_gboost),
    ],
    final_estimator=Ridge(random_state=42),  
    cv=5,
    n_jobs=-1,
    passthrough=False
)

# Grid for Ridge alpha
param_grid = {
    "final_estimator__alpha": [0.001,0.1, 0.5,1.0, 5.0]
}
grid = GridSearchCV(
    estimator=base_stack,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

# Run grid search
grid.fit(X_train, y_train)

# Best alpha 
best_alpha = grid.best_params_["final_estimator__alpha"]
best_cv_rmse = np.sqrt(-grid.best_score_)


best_stack = grid.best_estimator_
y_pred_test = best_stack.predict(X_test)
rmse_stack = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f"Stacking RMSE: {rmse_stack:.4f}")




Fitting 3 folds for each of 5 candidates, totalling 15 fits
Stacking RMSE: 44.3641


## Final Analysis

### Comparison

In [ ]:
import pandas as pd

rmse_data = {
    "Model": [
        "Baseline (Linear Regression)",
        "Bagging Regressor",
        "Gradient Boosting Regressor",
        "Stacking Regressor"
    ],
    "RMSE": [
        rmse_lr,
        rmse_bagging,
        rmse_gboost,
        rmse_stack
    ]
}

rmse_table = pd.DataFrame(rmse_data)

rmse_table = rmse_table.sort_values(by="RMSE").reset_index(drop=True)

rmse_table.style.hide(axis="index") \
    .set_caption("Comparative RMSE of All Models") \
    .format({"RMSE": "{:.2f}"})


Model,RMSE
Stacking Regressor,44.36
Gradient Boosting Regressor,44.43
Baseline (Linear Regression),100.45
Bagging Regressor,101.46


### Conclusion

The best perorming model is **Stacking Regressor**.

The Stacking Regressor achieved the lowest RMSE slightly outperforming the Gradient Boosting Regressor while it significantly improved the baseline and bagging models.
This shows that combining diverse learners in stacking effectively balances bias and variance.

Linear Regression is a high-bias and low variance model. It assumes a strict linear relationship between predictors and the target variable hence it underfits the model.

Stacking on the other hand combines multiple models like KNN and bagging, gradient boosting that each handle bias and variance differently. The meta learner (Ridge Regression) then optimally combines these models to create a model with lower bias without drastically increasing variance.